# Fine-tune a small model for text-to-SQL (sales table)
1. **Runtime -> Change runtime type -> T4 GPU**
2. Run the cells in order. Upload `train.jsonl`, `val.jsonl`, `schema.txt` and `sales.db` when asked.
3. After any runtime restart, rerun every cell from the top.

In [ ]:
!pip uninstall -y torchao
!pip install -q transformers peft datasets accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT FOUND - change the runtime type")

In [ ]:
from google.colab import files
files.upload()   # select train.jsonl, val.jsonl, schema.txt and sales.db

In [ ]:
!ls

In [ ]:
import json

MODEL = "Qwen/Qwen2.5-Coder-0.5B"
SCHEMA = open("schema.txt", encoding="utf-8").read().strip()

def build_prompt(question):
    return f"### Table\n{SCHEMA}\n### Question\n{question}\n### SQL\n"

def load(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train, val = load("train.jsonl"), load("val.jsonl")
print(len(train), "train /", len(val), "val")
print(build_prompt(train[0]["question"]) + train[0]["sql"])

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)

def encode(example):
    prompt_ids = tok(build_prompt(example["question"]))["input_ids"]
    sql_ids = tok(example["sql"] + tok.eos_token)["input_ids"]
    return {
        "input_ids": prompt_ids + sql_ids,
        "attention_mask": [1] * (len(prompt_ids) + len(sql_ids)),
        "labels": [-100] * len(prompt_ids) + sql_ids,   # learn only the SQL part
    }

train_ds = Dataset.from_list(train).map(encode, remove_columns=["question", "sql"])
print("longest example:", max(len(x) for x in train_ds["input_ids"]), "tokens")

In [ ]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

# fp32 weights + fp16 mixed precision is the safe combination on a free T4
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="out", num_train_epochs=3, per_device_train_batch_size=8, #USE per_device_train_batch_size=16, FOR BETTER RESULT
        learning_rate=2e-4, fp16=True, logging_steps=50,
        save_strategy="no", report_to="none",
    ),
    train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
)
trainer.train()   # roughly 8-15 minutes on a T4

In [ ]:
merged = model.merge_and_unload().eval()

def generate(question):
    inputs = tok(build_prompt(question), return_tensors="pt").to(merged.device)
    with torch.no_grad():
        out = merged.generate(**inputs, max_new_tokens=160, do_sample=False,
                              pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate("which city had the highest book sales?"))

In [ ]:
# Hand-written questions the generator never produced - this is the honest test
HAND_TEST = [
 {
  "question": "which city had the highest book sales?",
  "sql": "SELECT customer_city, SUM(total_inr) AS revenue FROM sales WHERE category = 'Books' GROUP BY customer_city ORDER BY revenue DESC LIMIT 1;"
 },
 {
  "question": "total sales of Smartphone in 2025",
  "sql": "SELECT SUM(total_inr) FROM sales WHERE product = 'Smartphone' AND strftime('%Y', order_date) = '2025';"
 },
 {
  "question": "how many orders used UPI in Delhi?",
  "sql": "SELECT COUNT(*) FROM sales WHERE customer_city = 'Delhi' AND payment_mode = 'UPI';"
 },
 {
  "question": "average discount on Clothing orders",
  "sql": "SELECT AVG(discount_pct) FROM sales WHERE category = 'Clothing';"
 },
 {
  "question": "units of Jeans sold in Mumbai",
  "sql": "SELECT SUM(quantity) FROM sales WHERE customer_city = 'Mumbai' AND product = 'Jeans';"
 },
 {
  "question": "revenue by payment mode in 2024",
  "sql": "SELECT payment_mode, SUM(total_inr) AS revenue FROM sales WHERE strftime('%Y', order_date) = '2024' GROUP BY payment_mode ORDER BY revenue DESC;"
 },
 {
  "question": "which product earned the least?",
  "sql": "SELECT product, SUM(total_inr) AS revenue FROM sales GROUP BY product ORDER BY revenue ASC LIMIT 1;"
 },
 {
  "question": "top three cities by number of orders",
  "sql": "SELECT customer_city, COUNT(*) AS orders FROM sales GROUP BY customer_city ORDER BY orders DESC LIMIT 3;"
 },
 {
  "question": "average order value for Electronics in Pune",
  "sql": "SELECT AVG(total_inr) FROM sales WHERE customer_city = 'Pune' AND category = 'Electronics';"
 },
 {
  "question": "how many orders were placed in March 2025?",
  "sql": "SELECT COUNT(*) FROM sales WHERE strftime('%Y-%m', order_date) = '2025-03';"
 },
 {
  "question": "revenue from Groceries paid by Cash on Delivery",
  "sql": "SELECT SUM(total_inr) FROM sales WHERE category = 'Groceries' AND payment_mode = 'Cash on Delivery';"
 },
 {
  "question": "which category gives the biggest average discount?",
  "sql": "SELECT category, AVG(discount_pct) AS avg_discount FROM sales GROUP BY category ORDER BY avg_discount DESC LIMIT 1;"
 },
 {
  "question": "monthly revenue for Books in 2025",
  "sql": "SELECT strftime('%Y-%m', order_date) AS month, SUM(total_inr) AS revenue FROM sales WHERE category = 'Books' AND strftime('%Y', order_date) = '2025' GROUP BY month ORDER BY month;"
 },
 {
  "question": "how many units did each category sell?",
  "sql": "SELECT category, SUM(quantity) AS units FROM sales GROUP BY category ORDER BY units DESC;"
 },
 {
  "question": "orders with discount above 10% in Chennai",
  "sql": "SELECT COUNT(*) FROM sales WHERE customer_city = 'Chennai' AND discount_pct > 10;"
 },
 {
  "question": "total revenue in Kanpur",
  "sql": "SELECT SUM(total_inr) FROM sales WHERE customer_city = 'Kanpur';"
 },
 {
  "question": "best selling product in Jaipur by units",
  "sql": "SELECT product, SUM(quantity) AS units FROM sales WHERE customer_city = 'Jaipur' GROUP BY product ORDER BY units DESC LIMIT 1;"
 },
 {
  "question": "which city has the lowest revenue for Electronics?",
  "sql": "SELECT customer_city, SUM(total_inr) AS revenue FROM sales WHERE category = 'Electronics' GROUP BY customer_city ORDER BY revenue ASC LIMIT 1;"
 }
]

In [ ]:
import sqlite3

def run(sql):
    con = sqlite3.connect("sales.db")
    try:
        return sorted(con.execute(sql).fetchall(), key=str)
    except Exception as e:
        return f"ERROR: {e}"
    finally:
        con.close()

ok = 0
for ex in HAND_TEST:
    pred = generate(ex["question"])
    good = run(pred) == run(ex["sql"])
    ok += good
    print("OK  " if good else "FAIL", ex["question"])
    if not good:
        print("     got :", pred)
        print("     want:", ex["sql"])
print(f"\nHand-written test: {ok}/{len(HAND_TEST)} correct")

sample = val[:100]
correct = sum(run(generate(ex["question"])) == run(ex["sql"]) for ex in sample)
print(f"Validation accuracy (same generator as training, so it looks optimistic): {correct / len(sample):.0%}")

In [ ]:
from huggingface_hub import login
login()   # paste a Hugging Face token with WRITE access (Settings -> Access Tokens)

REPO = "YOUR_USERNAME/sales-text2sql-qwen-0.5b"   # <- change this
merged.push_to_hub(REPO)
tok.push_to_hub(REPO)
print("Uploaded:", REPO)